In [15]:
# BLOCK RELATED TO SEARCH
from minsearch import Index

import minsearch
import json

with open('parse/documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

documents = []
for dict_course in docs_raw:
    course = dict_course['course']
    for dict_document in dict_course['documents']:
        documents.append({ 
            'course': course,
            'text': dict_document['text'],
            'section': dict_document['section'],
            'question': dict_document['question']
        })
        
index = Index(
    text_fields=["question","text","section"],
    keyword_fields=["course"]
)
index.fit(documents)
filter_dict = {"course": "data-engineering-zoomcamp"}
results = index.search(
    query=query, 
    boost_dict={'question': 3.0, 'section':0.5},
    filter_dict=filter_dict,
    num_results=3
)

In [6]:
from openai import OpenAI

In [3]:
client = OpenAI(api_key="")

In [7]:
query = "Can I join the course if it has already started?"

In [9]:
response = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {"role":"user", "content":query }
    ]
)

In [13]:
response.choices[0].message.content

'Whether you can join a course after it has started depends on the specific policies of the institution or organization offering the course. Here are a few general guidelines to consider:\n\n1. **Institutional Policy**: Some institutions allow late enrollment within a certain period after the course has started, while others have strict deadlines.\n\n2. **Course Type**: For self-paced online courses, late enrollment might be more flexible. However, for in-person or synchronous online courses, catching up might be more challenging.\n\n3. **Instructor Approval**: In some cases, obtaining approval from the course instructor or coordinator might be necessary for late enrollment.\n\n4. **Material Accessibility**: Consider if the course materials (recorded lectures, notes, etc.) are accessible to help you catch up on what you’ve missed.\n\n5. **Workload**: Evaluate your ability to catch up with the required work, assignments, and possibly missed assessments.\n\nTo find out for sure, you shou

In [37]:
prompt_template = f"""
You are a course teaching assistant. Answer the QUESTION based on the CONTEXT. Use only the facts from the CONTEXT when answering the QUESTION.If the CONTEXT doesn't contain the answer output NONE.
QUESTION: {query}

CONTEXT: 
{context}
""".strip()

In [16]:
results

[{'course': 'data-engineering-zoomcamp',
  'text': "Yes, even if you don't register, you're still eligible to submit the homework.\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?'},
 {'course': 'data-engineering-zoomcamp',
  'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?'},
 {'course': 'data-engineering-zoomcamp',
  'text': 'Yes, the slack channel remains open and you can ask questions there. But always sear

In [38]:
context = ""
for row_dict in results:
    context = context + f"section: {row_dict['section']}\nquestion: {row_dict['question']}\nanswer: {row_dict['text']}\n\n"

In [39]:
print(context)

section: General course-related questions
question: Course - Can I still join the course after the start date?
answer: Yes, even if you don't register, you're still eligible to submit the homework.
Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

section: General course-related questions
question: Course - Can I follow the course after it finishes?
answer: Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.
You can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.

section: General course-related questions
question: Course - Can I get support if I take the course in the self-paced mode?
answer: Yes, the slack channel remains open and you can ask questions there. But always search the channel first and second, check the

In [42]:
prompt = prompt_template.format(query=query, context=context).strip()

In [43]:
print(prompt)

You are a course teaching assistant. Answer the QUESTION based on the CONTEXT. Use only the facts from the CONTEXT when answering the QUESTION.If the CONTEXT doesn't contain the answer output NONE.
QUESTION: Can I join the course if it has already started?

CONTEXT: 
section: General course-related questions
question: Course - Can I still join the course after the start date?
answer: Yes, even if you don't register, you're still eligible to submit the homework.
Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

section: General course-related questions
question: Course - Can I follow the course after it finishes?
answer: Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.
You can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone 

In [45]:
response = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {"role":"user", "content":prompt }
    ]
)

In [46]:
response.choices[0].message.content

'Yes, you can still join the course after it has started. You are eligible to submit the homework, but be mindful of the deadlines for turning in assignments and the final projects.'